# Iterables vs. Iterators vs. Generators

## __`itertools`__
* functions for efficient looping
  * all of its functions return iterators
  * some produce finite sequences
  * others produce infinite sequences

In [ ]:
from itertools import zip_longest # cf. built-in zip()
letters = ['a', 'b', 'c', 'd']
fruits = ['apple', 'banana', 'cherry']

for item1, item2 in zip_longest(letters, fruits, fillvalue='***'):
    print(item1, '=>', item2)

In [ ]:
from itertools import count

counter = count(start=789)

for _ in range(10):
    print(next(counter))

In [ ]:
names = 'Alice Bob Charlie'.split()

for index, name in zip(count(101), names):
    print(index, name)

In [ ]:
from itertools import count

counter = count(1, 0.25)

for _ in range(10):
    print(next(counter))

In [ ]:
from itertools import cycle

sizes = ['S', 'M', 'L']
sc = cycle(sizes)

for _ in range(15):
    print(next(sc), end=' ')

In [ ]:
from itertools import count, islice

for num in islice(count(1, 0.25), 13, 17):
    print(num)

In [3]:
# some produce a finite sequence from an infinite sequence
from itertools import islice, cycle

colors = cycle(['red', 'white', 'blue']) # infinite
limited = islice(colors, 5, 10)

for color in limited:
    print(color, end= ' ')

blue red white blue red 

In [ ]:
from itertools import chain
# create a new iterable which chains together 2+ iterables

rank = list(range(2, 11))
picture = tuple('JQKA')
    
list(chain(rank, picture))

In [ ]:
from itertools import repeat
# repeats a value (in)finitely
list(repeat(1, 5))

In [ ]:
from itertools import permutations, combinations, combinations_with_replacement

# How many 4-digit numbers use only 1, 2, 3, 4, 5, and 6?
# Note we want ALL possible orderings and order matters (1234 and 4321 are different)
digits = list(range(1, 7))
len(list(permutations(digits, 4)))

In [ ]:
list(combinations(digits, 4)) # all possible groupings, independent of order

In [ ]:
list(combinations_with_replacement(digits, 4)) # all possible groupings where items can repeat
# (n + k - 1)! / (k! x (n - 1)!)

## Now, let's explore the differences between...
* a container
* an iterable
* an iterator
* a generator
* a generator expression
* a {list, set, dict} comprehension


![alt-text](iterable-relationships.png "iterable relationships")

## Containers 
* data structures which hold elements
* support membership tests
* live in memory
* typically hold all their values in memory
* e.g., str, list, tuple, set, dict
* an object is a container when it can be asked whether it _contains_ a certain element

__`[x for x in ...]`__, __`{x for x in ...}`__, __`{k: v for ...}`__

* eager
* all values at once

In [ ]:
1 in [1, 2, 3], 0 in [1, 2, 3]

In [ ]:
4 in {4, 5, 6}, 1 in {4, 5, 6}

In [ ]:
38 in ('Colorado', 'Denver', 38, 6_012_561)

In [ ]:
# for dicts, membership checks the keys, not the values
'Colorado' in { 'California': 31, 'Colorado': 38}, 31 in { 'California': 31, 'Colorado': 38}

In [ ]:
'J' in 'Steve Jobs', 'Job' in 'Steve Jobs', 'Jobs' not in 'Carlos Jobim'

## Iterables
* any object, not necessarily a data structure, that can return an iterator (with the purpose of returning all of its elements)
* the __`__iter__()`__ function returns an iterator
    * ...therefore, any object which has the __`__iter__()`__ method is an iterable
* most containers are also iterable
* many more things are iterable as well (e.g., open files, open sockets, etc.)

In [ ]:
mylist = [1, 2, 3]
listiter1 = iter(mylist)
print(type(listiter1))

In [ ]:
listiter2 = mylist.__iter__() # iter() maps to __iter__()
print(type(listiter2))

In [ ]:
next(listiter1)

In [ ]:
listiter1.__next__() # next() maps to __next__()

In [ ]:
next(listiter1)

In [ ]:
next(listiter1)

In [ ]:
next(listiter2)

In [ ]:
type(mylist), type(listiter1), type(listiter2)

In [ ]:
# a list is iterable, but it is not its own iterator
next(mylist)

In [ ]:
iter(mylist) is mylist

In [ ]:
mylistiter = iter(mylist)
print(f'0x{id(mylist):x}, 0x{id(mylistiter):x}')

In [ ]:
# When we write...
mylist = [1, 2, 3]

for x in mylist:
    ...
# ...this is what happens

![alt-text](iterable.png "iterable")

## We can see this by disassembling the Python code...

In [ ]:
import dis

mylist = [1, 2, 3]
total = 0
dis.dis('for item in mylist: total += item')

In [ ]:
# the for loop is doing this under the hood...
it = iter(mylist)
total = 0

while True:
    try:
        item = next(it)
        print('next() produced:', item)
    except StopIteration:
        print('StopIteration received')
        break
    total += item

print(total)

## So what is an iterator?
* a stateful object that produces the next value when you call __`next()`__ on it
* any object that has a __\_\_`next`\_\_()__ method is therefore an iterator
* how it produces a value is irrelevant
* in other words, an iterator is a value factory
 * each time you ask it for "the next" value, it knows how to compute it because it holds internal state

## Let's build our own iterator!

In [ ]:
class Fibonacci(object):
    def __init__(self):
        self.prev = 0
        self.curr = 1

    def __iter__(self):
        return self

    def __next__(self):
        """ each call to next() does two important things:
        1. modify its state for the subsequent next() call
        2. produces a result for the current call
        """
        value = self.curr
        self.curr += self.prev
        self.prev = value
        
        if value > 1000:
            raise StopIteration
        return value

# Note that this class is both an iterable due to __iter__()
# method and its own iterator, due to __next__() method!

f = Fibonacci()
print(next(f), next(f), 'before the for loop')

for num in f:
    print(num, end=' ')

In [ ]:
from collections import namedtuple 

card = namedtuple('Card', ['rank', 'suit'])

class DeckOfCards:
    ranks = list(range(2, 11)) + list('JQKA')
    suits = 'clubs diamonds hearts spades'.split()

    def __init__(self):
        self._cards = [card(rank, suit) for suit in self.suits for rank in self.ranks]
    
    def __str__(self):
        time_to_print = [str(x.rank) + ' of ' + str(x.suit) for x in self._cards]
        return str('\n'.join(time_to_print))
    
    # use the iterator from the underlying list
    # instead of relying on the "default" Python iterator
    # which makes use of __getitem__ and __len__ (if it exists)
    # (__getitem__ should be enough)
    
    def __iter__(self):
        return iter(self._cards)

deck = DeckOfCards()

## Lab: Iterators

Write your own iterator class which takes an iterable and each time it's invoked, it returns a *random* element. The iterator should stop (i.e., __`raise`__ the __`StopIteration`__ exception) when it has returned all elements of the iterable.

Example: __`MyRandomIterator([1, 2, 3])`__ might return

`
2
3
1
...then raise StopIteration`

In [ ]:
for card in deck:
    print(card)

## Generators
* a generator allows you to write iterators much like the Fibonacci iterator above but in an elegant, succinct syntax that avoids writing classes with __\_\_`iter`\_\_`()`__ and __\_\_`next`\_\_`()`__ methods
* works with __`next()`__
* every generator is an iterator (but not vice versa!) 
* a generator is a factory that lazily produces values (i.e, one value at a time)
* two types: generator _functions_ and generator _expressions_

## The __`yield`__ statement
* before we jump into generators, let's take an in-depth look at what makes them possible...
* when a normal Python function is invoked, execution starts at the first line and continues until a __`return`__ statement is encountered or an exception is thrown (remember that "falling off the end of the function" is the same as if we had written __`return None`__)
    * once a function returns, that's it–any work done by the function and stored in local variables is lost
    * the next call to the function starts everything anew
* there are times when we'd like to have a "function" which yields a series of values, i.e., it would have to save its state to that the next time it's invoked, it picks up where it left off
    * we use the term "yield" here because in fact we are *not returning* to the caller i.e., we are not returning control of execution to the point where the function was called
    * instead of __`return`__-ing, we are __`yield`__-ing, which implies that the transfer of control is temporary and voluntary–our function expects to regain control in the future
* functions that use __`yield`__ instead of __`return`__ are generator functions (or *coroutines* in other languages)
* think of __`yield`__ as __`return`__ + "some magic" for generator functions

### So what's the magic?
* when __`yield`__ is called the state of the generator function is recorded
    * the value of all variables are saved
    * the next line of code to be executed is also saved
    * i.e., the function simply resumes where it left off

In [ ]:
def simple_generator():
    yield 1
    yield 'boo!'
    yield 3
    
for value in simple_generator():
    print(value)

## Why do we need generator functions?
* initially, an easy way to write code that produced a series of values
    * without them, writing something like a random number generator required a class or module that  generated values and kept track of state between calls
    * with them, doing the above is greatly simplified

#### suppose we want a function which, given a list of numbers, returns a list of those numbers which are prime:

In [ ]:
def get_primes(nums):
    return ([num for num in nums if is_prime(num)])

* let's write __`is_prime()`__ so we can test...

In [ ]:
def is_prime(num):
    import math
    
    if num > 2 and num % 2 == 0:
       return False
        
    for possible_divisor in range(3, int(math.sqrt(num)) + 1, 2):
        if num % possible_divisor == 0:
            return False
            
    return True

In [ ]:
get_primes(list(range(2, 50)))

* now suppose we want to use it for _large_ lists of numbers...so large, in fact, that they won't fit in memory
    * so now we want the function to take a starting value, and return all the primes that are greater than that value
    * since functions only return once, they only have one "chance" to return a value (or list of values)
    * what if our function could return the *next* value, rather than a list?
        * we wouldn't need to create a list at all!

## What is a generator function?
* defined like a normal function, but whenever it needs to generate a value, it does so with the __`yield`__ keyword rather than __`return`__
    * if the body of a def contains __`yield`__, the function automatically becomes a generator function (even if it also has a __`return`__ statement)
    * ...there's nothing else we need to do to create one
* generator functions create *generator iterators* (or simply, a *generator*)
    * a generator is a special type of iterator (meaning it has a __\_\_`next`\_\_`()`__ function)
    * to get the next value from a generator, we use the same built-in function as for iterators: __`next()`__
* let's return to the more basic notion of a generator function...

### Now we can rewrite __`get_primes()`__ as a generator...

In [ ]:
def get_primes(num):
    while True:
        if is_prime(num):
            yield num
        num += 1

In [ ]:
prime_generator = get_primes(2)

for _ in range(10):
    print(next(prime_generator), end=' ')

In [ ]:
from itertools import islice
# islice returns a specified slice of an iterator without consuming the rest

for prime in islice(get_primes(2), 10):
    print(prime, end=' ')

In [ ]:
for prime in islice(get_primes(2), 10, 20):
    print(prime, end=' ')

#### (Note that if a generator function calls return (or simply hits the end of the function), then a __`StopIteration`__ exception is raised, signaling the generator is exhausted, just as an iterator does)

In [ ]:
# let's try the Fibonacci sequence as a generator

def fibonacci():
    """defined as a normal function, but...no return keyword
    
    The yield keyword returns a value, but the function retains its state.
    """
    prev, curr = 0, 1
    
    while True:
        yield curr
        prev, curr = curr, prev + curr

In [ ]:
import random

f = fibonacci()
print(next(f), next(f), 'before the for loop', sep='\n')

for num in range(0, random.randint(10, 50)):
    val = next(f)
    print(val, end=' ')

In [ ]:
for num in islice(fibonacci(), 1, random.randint(2, 50)):
    print(num, end=' ')

## Sending values into generators
* we can pass values *into* generators using the __`send()`__ function
* let's go back to the prime number example but instead of printing every prime number > some number, we'll find the smallest prime number greater than successive powers of a number (i.e. for 10, we want the smallest prime greater than 10, then 100, then 1000, etc.)

In [ ]:
def get_primes(num):
    while True:
        if is_prime(num):
            # 1. yield num
            # 2. wait for a value to be sent to me
            # 3. set num to that value
            num = yield num 
        num += 1

* and we can print the next prime greater than 10, 100, 1000, as follows:

In [ ]:
def print_successive_primes(iterations, base=10):
    prime_generator = get_primes(base)
    prime_generator.send(None)
    
    for power in range(iterations):
        print(prime_generator.send(base ** power))

In [ ]:
print_successive_primes(15)

* printing __`generator.send()`__ is possible because __`send`__ both sends a value to the generator and returns the value yielded by the generator
* note that the first time we send a value into a generator, it must be __`None`__

## Now let's look at a generator _expression_
* generator equivalent of a list comprehension

In [5]:
squares = [num * num for num in range(1, 11)] # list comprehension
squares

[1, 4, 9, 16, 25, 36, 49, 64, 81, 100]

In [6]:
squares = {num * num for num in range(1, 11)} # set comprehension
squares

{1, 4, 9, 16, 25, 36, 49, 64, 81, 100}

In [7]:
squares = {num: num * num for num in range(1, 11)} # dict comprehension
squares

{1: 1, 2: 4, 3: 9, 4: 16, 5: 25, 6: 36, 7: 49, 8: 64, 9: 81, 10: 100}

In [8]:
squares = (num * num for num in range(1, 11)) # generator expression (NOT a 'tuple comprehension')
squares

<generator object <genexpr> at 0x109bbd150>

In [9]:
next(squares), next(squares)

(1, 4)

In [10]:
list(squares) # for thing in squares: print(thing)

[9, 16, 25, 36, 49, 64, 81, 100]

## Lab: Generators
* modify your random iterator to be a generator function